# PyTorch Roster Classifier — Reusable Template

**Short name:** `PTEdu` template. Point `DATA_PATH` at a tidy student table, edit `FEATURE_COLS` / `TARGET_COL` / `CLASS_ORDER`, keep the scale step.

Contract:

1. Features `float32`, class indices `long`.
2. Train-only z-score when columns live on different units.
3. `nn.Module` with a ReLU hidden layer, logits out.
4. `CrossEntropyLoss` + Adam.
5. `model.eval()` + `torch.no_grad()` before you score.
6. Simulation knobs: hidden / lr / epochs / label noise / scale.

Not a grading engine.


## Inline cheat-sheet (keep this cell visible)

See also **`PTEdu_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Roster row | one student; columns = attendance %, study h/week, prior GPA, assignment avg |
| Bands | `support=0`, `on_track=1`, `honors=2` |
| Tensor ranks | scalar 0-D, vector 1-D, matrix 2-D |
| From a list | `torch.tensor([[1,2],[3,4]])` |
| Factories | `zeros`, `ones`, `arange` (end exclusive), `linspace` (both ends in), `*_like` |
| dtypes | features `float32`; class indices `long` |
| Scale | train mean/sd only — units are `%`, hours, 4.0 GPA, 100-pt avg |
| Module | `OutcomeNet`: `4 → hidden ReLU → 3` logits |
| Call | `model(x)` — **not** `model.forward(x)` |
| Loss | `nn.CrossEntropyLoss()` — do **not** softmax first |
| Train step | `zero_grad()` → forward → loss → `backward()` → `step()` |
| Eval | `model.eval()` **and** `with torch.no_grad():` |
| Labels | `torch.argmax(logits, dim=1)` |

**Education-specific gotcha:** raw units underfit (≈69% here). Scaled features reach ≈97%. Iris hid this because every column was centimetres.


## Drop-in training script


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DATA_PATH = "data/students_outcomes.csv"
FEATURE_COLS = ["attendance_pct", "study_hours", "prior_gpa", "assignment_avg"]
TARGET_COL = "outcome"
CLASS_ORDER = ["support", "on_track", "honors"]
HIDDEN = 16
LR = 0.01
EPOCHS = 150
TEST_SIZE = 0.2
SEED = 42
SCALE = True

df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLS].to_numpy(np.float32)
y = df[TARGET_COL].map({c: i for i, c in enumerate(CLASS_ORDER)}).to_numpy(np.int64)
assert not np.isnan(y).any(), "CLASS_ORDER does not cover every label"

def split(X, y, test_size=0.2, seed=42):
    rng = np.random.RandomState(seed)
    tr, te = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_te = int(round(len(idx) * test_size))
        te.append(idx[:n_te]); tr.append(idx[n_te:])
    tr, te = np.concatenate(tr), np.concatenate(te)
    rng.shuffle(tr); rng.shuffle(te)
    return X[tr], X[te], y[tr], y[te]

Xtr, Xte, ytr, yte = split(X, y, TEST_SIZE, SEED)
if SCALE:
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

Xtr_t, Xte_t = torch.tensor(Xtr), torch.tensor(Xte)
ytr_t, yte_t = torch.tensor(ytr, dtype=torch.long), torch.tensor(yte, dtype=torch.long)

class MLP(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_h)
        self.fc2 = nn.Linear(d_h, d_out)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

torch.manual_seed(SEED)
model = MLP(X.shape[1], HIDDEN, len(CLASS_ORDER))
opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
losses = []
for ep in range(EPOCHS):
    opt.zero_grad()
    loss = crit(model(Xtr_t), ytr_t)
    loss.backward(); opt.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    pred = model(Xte_t).argmax(1)
acc = (pred == yte_t).float().mean().item()
print(f"test acc {acc*100:.2f}%  final loss {losses[-1]:.4f}  scale={SCALE}")
plt.plot(losses); plt.title("loss"); plt.xlabel("epoch"); plt.show()


## How to reuse

1. Point `DATA_PATH` at a tidy student CSV.
2. List numeric columns. Encode the target as `0..K-1`.
3. Keep `SCALE = True` whenever units differ (GPA vs percent vs hours).
4. `CrossEntropyLoss` for exclusive bands. For a single at-risk logit use `BCEWithLogitsLoss`.
5. When *n* grows, wrap tensors in a `DataLoader`.
6. Do not ship the scores as grades.

Short name stays `PTEdu`.
